# DPD replay — ADRV9026

Every row of a capture CSV is replayed at its saved LO, TX attenuation and ORx
gain. The ORx AGC does not run. The DPD file for the row is transmitted as
stored: no peak normalize, no power change.

Each capture is aligned to the waveform that was transmitted. NMSE, PAPR
compression and ACLR are against the original input of that row. The DUT
captures and `{LABEL}.csv` go to `OUT_DIR`.

DPD files hold the waveform times `FILE_SCALE`: 1.0 for normalized files, 2048
for files written in TX codes scaled by `2**11`.

## 1. Parameters

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILE = "ADRV9025Init_StdUseCase98_LinkSharing.profile"

TX_CHANNEL  = TxChannel.TX1
ORX_CHANNEL = RxChannel.ORX1

MODEL = "gmp"
CONDITIONS_CSV = "captures/operating_sweep/TX1_conditions.csv"
DPD_DIR = f"DPD/{MODEL}"
DPD_FILE = "{name}_dpd_" + MODEL + ".txt"     # per-row file name inside DPD_DIR
FILE_SCALE = 1.0
LABEL = f"dpd_{MODEL}"
OUT_DIR = f"DPD_DUT/{MODEL}"
OUTPUT_OVERSAMPLE = 2

## 2. Imports, config, profile

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import full_scale

from adrvtrx.conditions import load_conditions, load_dut_records
from adrvtrx.replay import replay_conditions

cfg = load_config()
cfg.profile_name = PROFILE
info = read_profile(cfg.profile_path)
FS = info.orx_rate_hz
rows = load_conditions(CONDITIONS_CSV)
cfg.lo.lo1_hz = int(rows[0].freq_hz)
print("Profile:", cfg.profile_path.name)
print(f"{len(rows)} conditions in {CONDITIONS_CSV}   DPD files {DPD_DIR}/{DPD_FILE}")

## 3. Connect and program

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()

radio.set_tx_atten(TX_CHANNEL, 15.0)
lock = radio.retune_lo("LO1", int(cfg.lo.lo1_hz))

for k, v in verify_status(radio).items():
    print(f"{k}: {v}")
print(f"LO1 = {radio.get_lo('LO1')/1e6:.3f} MHz   lock 0x{lock:X}")
print(f"TX1 atten = {radio.get_tx_atten(TX_CHANNEL):.2f} dB")

## 4. Replay every condition

All files are checked before the first transmission. TX stops between
frequencies and when the replay ends or fails.

In [ ]:
def waveform_for(row):
    return Path(DPD_DIR) / DPD_FILE.format(name=row.name)

def show(rec):
    print(f"{rec.name}  atten {rec.tx_atten_db:.2f} dB  gain {rec.orx_gain}  "
          f"NMSE {rec.nmse_db:.2f} dB  PAPR comp {rec.papr_compression_db:.2f} dB  "
          f"ACLR {rec.aclr_lower_dbc:.2f}/{rec.aclr_upper_dbc:.2f} dBc  "
          f"railed {rec.railed}  tx clipped {rec.tx_clipped}")

dut_csv = replay_conditions(
    radio, CONDITIONS_CSV, waveform_for,
    tx=TX_CHANNEL, orx=ORX_CHANNEL, tx_bits=info.tx_bits, rx_bits=info.rx_bits, fs=FS,
    out_dir=OUT_DIR, label=LABEL, oversample=OUTPUT_OVERSAMPLE, file_scale=FILE_SCALE,
    on_row=show,
)
print(dut_csv)

## Safe-state and disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")

## 6. Before and after, joined on `name`

In [ ]:
def worst(r):
    return max(r.aclr_lower_dbc, r.aclr_upper_dbc)

before = {r.name: r for r in load_conditions(CONDITIONS_CSV)}
after = load_dut_records(dut_csv)
keys = sorted({(r.bw_mhz, r.freq_hz) for r in after})
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for i, (bw, freq) in enumerate(keys):
    grp = sorted((r for r in after if r.bw_mhz == bw and r.freq_hz == freq),
                 key=lambda r: r.backoff_db)
    b = [r.backoff_db for r in grp]
    c = f"C{i % 10}"
    label = f"{bw} MHz @ {freq / 1e6:.0f} MHz"
    ax[0].plot(b, [before[r.name].nmse_db for r in grp], "o--", c=c, alpha=0.5)
    ax[0].plot(b, [r.nmse_db for r in grp], "o-", c=c, label=label)
    ax[1].plot(b, [worst(before[r.name]) for r in grp], "o--", c=c, alpha=0.5)
    ax[1].plot(b, [worst(r) for r in grp], "o-", c=c, label=label)
ax[0].set_ylabel("NMSE (dB)"); ax[1].set_ylabel("worst ACLR (dBc)")
for a in ax:
    a.set_xlabel("backoff (dB)"); a.grid(True)
ax[0].set_title("dashed: without DPD, solid: with DPD")
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()